# 02 · Next-token prediction by hand
Level L1 · Part 1 · core · ~25 min · builds on 01

## Goal
By the end of this lesson you can generate text one token at a time with a toy model you built
yourself, and say which of three **stop conditions** ended each run.

## The idea
In lesson 01 the model was a black box: text in, text out. Inside, a language model does one
small job: given the text so far, it gives a score to every possible **next token**.
A token is a piece of text; here a word or a punctuation mark (lesson 03 shows real tokens).
To write a sentence, code picks one token, appends it, and asks again. This repeat-and-append
loop is **autoregressive generation**. It runs until a **stop condition**: the model emits an
end token, a length limit is hit, or a chosen stop text appears.

Analogy: phone autocomplete. Keep tapping the middle suggestion and a sentence appears, one word at a time.

## Picture
```mermaid
flowchart LR
    P[Text so far] --> M[Model: score every next token]
    M --> C[Pick one token]
    C --> S{Stop condition?}
    S -->|no: append it| P
    S -->|yes| O[Finished text + stop reason]
```

## Step by step
### 1. Scout's notes become training text
We reuse Scout's six notes from lesson 01. They are the only text our toy model will ever see.

In [1]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
print(len(NOTES), "notes of training text")

6 notes of training text


### 2. Split text into tokens
A model works on tokens, not characters. We use a simple rule: each word and each punctuation
mark is a token. We also add two special tokens: `<s>` marks the start of a note and `</s>` its end.
The end token is how the model learns *where text stops*.

In [2]:
import re

START, END = "<s>", "</s>"

def tokenize(text: str) -> list[str]:
    """Words (letters, digits, apostrophes) and single punctuation marks."""
    return re.findall(r"[\w']+|[.,!?]", text)

print(tokenize(NOTES[0]["text"]))
sequences = [[START] + tokenize(n["text"]) + [END] for n in NOTES]
print(sum(len(s) for s in sequences), "tokens in total")

['Mars', 'has', 'two', 'small', 'moons', ',', 'Phobos', 'and', 'Deimos', '.']
70 tokens in total


### 3. Train a bigram model
A **bigram model** is the smallest possible language model: it predicts the next token from only the
one token before it. Training means counting: after `Mars`, which tokens came next, and how often?
Real models look at thousands of earlier tokens with a neural network, but the output is the same kind
of thing: a score for each possible next token.

In [3]:
from collections import Counter, defaultdict

counts = defaultdict(Counter)
for seq in sequences:
    for prev, nxt in zip(seq, seq[1:]):
        counts[prev][nxt] += 1
counts = dict(counts)          # freeze: looking up an unseen token now fails loudly

print("after 'Mars':", dict(counts["Mars"]))
print("after 'in':  ", dict(counts["in"]))
print("vocabulary size:", len({t for s in sequences for t in s}))

after 'Mars': {'has': 1, 'lasts': 1, 'in': 1}
after 'in':   {'the': 1, 'February': 1}
vocabulary size: 49


### 4. The model: probabilities for the next token
Turning counts into fractions gives a **probability distribution**: every candidate next token
with a number, all adding up to 1. This function *is* our model. It does not write text; it only scores.

In [4]:
def next_token_probs(prev: str) -> dict[str, float]:
    """The model: given the last token, the probability of each possible next token."""
    c = counts[prev]
    total = sum(c.values())
    return {tok: n / total for tok, n in c.items()}

for tok, p in next_token_probs(START).items():
    print(f"{tok:>8}  {p:.2f}")

    Mars  0.17
   Venus  0.17
       A  0.17
 Jupiter  0.17
     The  0.17
Saturn's  0.17


Six notes start with six different words, so after `<s>` each has probability 1/6.

### 5. Pick a token
Code, not the model, chooses. The simplest rule is **greedy decoding**: take the highest-probability
token. `max` keeps the first one it sees on a tie, so runs are repeatable. Lesson 04 replaces this
with random sampling (temperature).

In [5]:
def pick_greedy(probs: dict[str, float]) -> str:
    return max(probs, key=probs.get)

print(pick_greedy(next_token_probs(START)))
print(pick_greedy(next_token_probs("Mars")))

Mars
has


### 6. Generate autoregressively, step by step
Now the loop: score, pick, append, and feed the new token back in. We start from a **prompt**,
`"The rover"`, and print each step so you can watch the text grow.

In [6]:
tokens = [START] + tokenize("The rover")
for step in range(1, 16):
    nxt = pick_greedy(next_token_probs(tokens[-1]))
    tokens.append(nxt)
    print(f"step {step:2}: + {nxt!r:16} -> {' '.join(tokens[1:])}")
    if nxt == END:
        break

step  1: + 'Perseverance'   -> The rover Perseverance
step  2: + 'landed'         -> The rover Perseverance landed
step  3: + 'on'             -> The rover Perseverance landed on
step  4: + 'Mars'           -> The rover Perseverance landed on Mars
step  5: + 'has'            -> The rover Perseverance landed on Mars has
step  6: + 'two'            -> The rover Perseverance landed on Mars has two
step  7: + 'small'          -> The rover Perseverance landed on Mars has two small
step  8: + 'moons'          -> The rover Perseverance landed on Mars has two small moons
step  9: + ','              -> The rover Perseverance landed on Mars has two small moons ,
step 10: + 'Phobos'         -> The rover Perseverance landed on Mars has two small moons , Phobos
step 11: + 'and'            -> The rover Perseverance landed on Mars has two small moons , Phobos and
step 12: + 'Deimos'         -> The rover Perseverance landed on Mars has two small moons , Phobos and Deimos
step 13: + '.'              ->

It took 14 steps and ended on `</s>`: it stopped on its own. Now read the output: "The rover Perseverance
landed on Mars **has two small moons**, Phobos and Deimos." That is not a real sentence. It is note 5
glued to note 1. The model only sees one word back, so after `Mars` it simply followed note 1.
The result is a run-on that no grammar or fact-checker would accept, and the model cannot tell.

### 7. Three stop conditions
Let's wrap the loop in a function that returns the text **and why it stopped**. Real APIs report the same
thing (lesson 13). Three reasons:
- `end_token`: the model picked `</s>`, its "I am done" token.
- `max_tokens`: we hit our length limit. Always set one.
- `stop_sequence`: a text we asked to stop at appeared (it is cut off from the output).

In [7]:
def generate(prompt: str, max_tokens: int = 20, stop: list[str] | None = None) -> tuple[str, str]:
    tokens = [START] + tokenize(prompt)
    new: list[str] = []
    for _ in range(max_tokens):
        nxt = pick_greedy(next_token_probs(tokens[-1]))
        if nxt == END:
            return " ".join(new), "end_token"
        if stop and nxt in stop:
            return " ".join(new), "stop_sequence"
        tokens.append(nxt)
        new.append(nxt)
    return " ".join(new), "max_tokens"

print(generate("Saturn's"))

('rings are mostly made of ice .', 'end_token')


`Saturn's` led straight through note 6 to the end token. Now a prompt that never reaches `</s>`:

In [8]:
text, reason = generate("Jupiter", max_tokens=15)
print(reason, "|", text)

max_tokens | is the largest planet in the largest planet in the largest planet in the largest


The model is stuck in a cycle: `in -> the -> largest -> planet -> in`. With greedy picking it would
repeat forever; only `max_tokens` saved us. A **stop sequence** gives a cleaner cut: stop at the first period.

In [9]:
for prompt in ["Jupiter", "A day", "Venus"]:
    text, reason = generate(prompt, max_tokens=15, stop=["."])
    print(f"{reason:14} | {prompt} {text}")

max_tokens     | Jupiter is the largest planet in the largest planet in the largest planet in the largest
stop_sequence  | A day on Mars has two small moons , Phobos and Deimos
stop_sequence  | Venus spins backwards compared with most planets


`Jupiter` still hits `max_tokens` (the cycle has no period). `A day` and `Venus` stop at the period,
which is not part of the output. Look at `A day`: "A day on Mars has two small moons". This one reads
like a normal sentence, and it is false: no note says it, and a day is not something that has moons.
Fluent text that states something invented is called a **hallucination**. Our toy produced one from
word-pair statistics alone; big models do it too, just with more convincing text.
Same model, three different reasons to stop: your code must check which one.

## At work
- **Every production call sets a token limit.** Output tokens cost money and time, one token at a time,
  so teams cap output per call and count tokens per turn. → lesson 12
- **Checking the stop reason is not optional.** A common bug is treating text cut off at `max_tokens`
  as a complete answer (half a JSON object, a truncated summary). Every API returns a stop reason,
  but the names vary by provider: Anthropic's API, for example, uses `end_turn`, `max_tokens`,
  `stop_sequence` and more. → lesson 13
- **Products show tokens as they are generated.** Because generation is token by token, chat apps
  stream partial text instead of making users wait for the whole answer. → lesson 11
- **Real models sample, not always pick the top token**, which is why the same prompt can give different
  answers. → lesson 04

## What just happened
- `tokenize` split notes into word and punctuation tokens, framed by `<s>` and `</s>`.
- Counting pairs gave a bigram model; `next_token_probs` turned counts into a probability for each next token.
- `pick_greedy` chose the top token; appending it and asking again (autoregression) built a sentence.
- "The rover ..." stitched two notes together (5 then 1) into a run-on: the model only saw one word back.
- "A day on Mars has two small moons" read fluently but was invented: a tiny hallucination.
- `generate` stopped for three reasons: `end_token` (Saturn's), `max_tokens` (Jupiter's cycle), `stop_sequence` (the period).

## Common mistakes
**1. Continuing after the end token.** If the loop doesn't check for `</s>`, it asks the model
what comes after "the end". Our model never saw anything after `</s>`, so it has no answer.

`Saturn's` reaches `</s>` after 8 new tokens; the 9th step asks for the tokens after `</s>`, and there are none.

In [10]:
def generate_no_end_check(prompt, max_tokens=20):
    tokens = [START] + tokenize(prompt)
    for _ in range(max_tokens):
        tokens.append(pick_greedy(next_token_probs(tokens[-1])))
    return " ".join(tokens[1:])

generate_no_end_check("Saturn's")

KeyError: '</s>'

Fix: check for the end token before appending, as `generate` does.

**2. A prompt ending in a token the model never saw.** `Pluto` is not in any note, so there are no counts after it.

In [11]:
generate("Tell me about Pluto")

KeyError: 'Pluto'

Fix for the toy: guard the last token before generating, and return a clear stop reason instead of crashing.
Real models never fail like this, because every text maps to known tokens (lesson 03 shows how);
they just predict less well on unfamiliar text.

In [12]:
def safe_generate(prompt: str, **kwargs) -> tuple[str, str]:
    last = ([START] + tokenize(prompt))[-1]
    if last not in counts:
        return "", "unknown_token"
    return generate(prompt, **kwargs)

print(safe_generate("Tell me about Pluto"))
print(safe_generate("Saturn's"))

('', 'unknown_token')
('rings are mostly made of ice .', 'end_token')


## Try it
1. Call `generate("Mars", max_tokens=3)` and `generate("Mars")`. Print both results and stop reasons.
2. Add a seventh note, "Earth is in the solar system.", retrain `counts`, and generate from
   `"Jupiter"` again with `max_tokens=15`. Does it still loop? Print `next_token_probs("the")` to explain why.

In [13]:
# your code here

## Key terms
- **next-token prediction** — the one job of a language model: given the text so far, give a probability to every possible next token.
- **token** — the unit of text a model reads and writes; in this lesson a word or punctuation mark (lesson 03 shows real ones).
- **autoregressive generation** — producing text by predicting one token, appending it, and feeding the longer text back in.
- **bigram model** — a toy language model that predicts the next token from only the previous token, using counts.
- **greedy decoding** — always picking the highest-probability next token.
- **hallucination** — fluent model output that states something invented or false.
- **end token** — a special token that marks the end of a text; when the model picks it, generation stops.
- **stop condition** — the rule that ends generation: end token, `max_tokens` limit, or a stop sequence.
- **stop sequence** — a text you give in advance; generation stops when it appears, and it is left out of the output.